In [2]:
import pandas as pd

df = pd.read_csv("/lakehouse/default/Files/online_retail_II.csv")

print(df.shape)
df.head()

(1067371, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [3]:
# Drop rows missing essential info — can't analyze a sale with no product code or date
df = df.dropna(subset=["StockCode", "InvoiceDate", "Quantity"])

# Negative quantities are returns/cancellations, not sales — remove them
df = df[df["Quantity"] > 0]

# The date column comes in as plain text — convert it to a real date
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

# Remove any exact duplicate rows
df = df.drop_duplicates()

print(f"Clean data: {df.shape[0]} rows remaining")


Clean data: 1010540 rows remaining


In [4]:
# We only care about the date, not the exact time of sale
df["SaleDate"] = df["InvoiceDate"].dt.date

# For each product, on each day, add up how many units were sold
daily_demand = df.groupby(["StockCode", "SaleDate"])["Quantity"].sum().reset_index()
daily_demand = daily_demand.rename(columns={"Quantity": "UnitsSold"})

daily_demand.head()

,StockCode,SaleDate,UnitsSold
0,10002,2009-12-01,12
1,10002,2009-12-03,6
2,10002,2009-12-04,73
3,10002,2009-12-06,49
4,10002,2009-12-07,2


In [5]:
# For each product: average daily sales, and how much that swings day-to-day
product_stats = daily_demand.groupby("StockCode")["UnitsSold"].agg(
    avg_daily_demand="mean",
    demand_std="std"
).reset_index()

# Products sold only once have no "swing" number (NaN) — treat that as zero
product_stats["demand_std"] = product_stats["demand_std"].fillna(0)

product_stats.head()

,StockCode,avg_daily_demand,demand_std
0,10002,39.163717,93.198020
1,10002R,1.333333,0.577350
2,10080,21.321429,36.359372
3,10109,4.000000,0.000000
4,10120,10.375000,12.868690


In [6]:
# Assumptions — stated clearly because the raw data has no lead-time/stock info:
LEAD_TIME_DAYS = 7    # days it takes to get new stock in
Z_SCORE = 1.65         # standard safety factor for ~95% service level

# Extra stock to buffer against unpredictable demand
product_stats["safety_stock"] = Z_SCORE * product_stats["demand_std"] * (LEAD_TIME_DAYS ** 0.5)

# Reorder point = expected demand during lead time + safety buffer
product_stats["reorder_point"] = (product_stats["avg_daily_demand"] * LEAD_TIME_DAYS) + product_stats["safety_stock"]

product_stats.head()

,StockCode,avg_daily_demand,demand_std,safety_stock,reorder_point
0,10002,39.163717,93.198020,406.854991,681.001009
1,10002R,1.333333,0.577350,2.520417,11.853750
2,10080,21.321429,36.359372,158.726465,307.976465
3,10109,4.000000,0.000000,0.000000,28.000000
4,10120,10.375000,12.868690,56.178132,128.803132


In [7]:
# NOTE: This dataset only has sales history, not live stock counts.
# We simulate a starting stock of "30 days of average demand" per product
# to demonstrate the risk logic. In a real company, this line would instead
# pull from an actual inventory/ERP system.
product_stats["current_stock_simulated"] = (product_stats["avg_daily_demand"] * 30).round()

# A product is "at risk" if its (simulated) stock is below its reorder point
product_stats["stockout_risk"] = product_stats["current_stock_simulated"] < product_stats["reorder_point"]

at_risk = product_stats[product_stats["stockout_risk"]]
print(f"{len(at_risk)} of {len(product_stats)} products flagged at stockout risk")

92 of 4985 products flagged at stockout risk


In [9]:
product_stats.to_csv("/lakehouse/default/Files/product_stockrisk_summary.csv", index=False)
print("Saved!")

Saved!
